# Exp 01: Lecture models

Card: [exp01_lecture_models.md](../../experiments/exp01_lecture_models.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import Config, CrossValidator
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-lecture-models-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
else:
    OVERRIDES = {}
CONFIG_DIR = ROOT / "sections" / "model" / "configs" / "exp01_lecture_models"
CONFIGS = {
    stem: Config.from_toml(CONFIG_DIR / f"{stem}.toml", **OVERRIDES)
    for stem in ["resnet18", "efficientnet_b0", "alexnet"]
}
FIGURES = ROOT / "sections" / "model" / "figures"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode; categorical slots in fixed order
MUTED = "#898781"
SERIES = dict(
    zip(
        [c.name for c in CONFIGS.values()],
        ["#eb6834", "#1baf7a", "#eda100"],
        strict=True,
    )
)
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## Training


In [ ]:
# filled per model; rerunning a model's cell overwrites its entry
results = {}


def fit(stem):
    result = CrossValidator(CONFIGS[stem]).run()
    results[result.config.name] = result

In [ ]:
fit("resnet18")

In [ ]:
fit("efficientnet_b0")

In [ ]:
fit("alexnet")

In [ ]:
fold_ap_df = pd.concat(
    {name: r.fold_ap for name, r in results.items()}, names=["model"]
)
first = next(iter(results.values()))
EXPERIMENT, FOLDS = first.config.experiment, first.config.folds
y_oof = first.labels  # same rows for every model

## Training curves


In [ ]:
panels = [
    ("Loss (weighted BCE)", "train_loss_epoch", "val_loss"),
    ("mAP", "train_map", "val_map"),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, train_key, val_key) in zip(axes, panels):
    for name, r in results.items():
        epochs = np.arange(1, r.config.epochs + 1)
        ax.plot(epochs, r.mean_curve(train_key), color=SERIES[name], ls=":")
        ax.plot(epochs, r.mean_curve(val_key), color=SERIES[name], label=name)
    ax.set(title=f"{title}, mean over {len(FOLDS)} folds", xlabel="epoch")
axes[1].plot([], [], color=MUTED, label="validation")
axes[1].plot([], [], color=MUTED, ls=":", label="train")
axes[1].legend(loc="lower right")
fig.tight_layout()
figures = {"exp01_lecture_models_training.png": fig}
plt.show()

## Precision-recall curves


In [ ]:
best = {name: r.scores.best_f1() for name, r in results.items()}
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.3 * len(CLASSES), 3.6), sharey=True
)
for c, (cls, ax) in enumerate(zip(CLASSES, axes)):
    y = y_oof[:, c]
    ax.axhline(y.mean(), color=MUTED, ls="--", lw=0.8)
    for name, r in results.items():
        precision, recall, _ = precision_recall_curve(y, r.probs[:, c])
        b = best[name].loc[cls]
        ap = average_precision_score(y, r.probs[:, c])
        ax.step(
            recall,
            precision,
            where="post",
            color=SERIES[name],
            label=f"AP {ap:.2f} · F1 {b.f1:.2f} @ t {b.threshold:.2f}",
        )
        ax.plot(
            b.recall, b.precision, "o", ms=7, color=SERIES[name], mec="white", mew=1.5
        )
    ax.set(
        xlim=(0, 1),
        ylim=(0, 1),
        xlabel="recall",
        title=f"{cls}  (n+ {y.sum():.0f}, prevalence {y.mean():.2f})",
    )
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.2), handlelength=1.2)
axes[0].set_ylabel("precision")
models = [plt.Line2D([], [], color=SERIES[name]) for name in results]
fig.legend(
    models,
    [f"{name}  macro F1 {best[name].f1.mean():.3f}" for name in results],
    title=f"pooled out-of-fold predictions, folds {FOLDS}; dot = best F1",
    loc="upper left",
    ncol=len(results),
    bbox_to_anchor=(0, 1.15),
    alignment="left",
)
fig.tight_layout()
figures["exp01_lecture_models_pr.png"] = fig
plt.show()

In [ ]:
f1_df = pd.DataFrame({name: b.f1 for name, b in best.items()}).T
f1_df["macro F1"] = f1_df.mean(axis=1)
f1_df = f1_df.sort_values("macro F1", ascending=False)
f1_df.round(3)

## cAP comparison


In [ ]:
cap = fold_ap_df.groupby("model", sort=False).mean().sort_values("mAP", ascending=False)
cap.round(3)

In [ ]:
# difference to leader counts only if it has same sign on both folds
fold_map = fold_ap_df["mAP"].unstack("fold").loc[cap.index]
paired = fold_map.add_prefix("mAP fold ").join(
    (fold_map - fold_map.iloc[0]).add_prefix("diff fold ")
)
paired["below leader on both folds"] = (fold_map < fold_map.iloc[0]).all(axis=1)
paired.round(3)

In [ ]:
with Experiment(EXPERIMENT).run("summary", {"val_mode": "fast"}) as run:
    mlflow.log_table(cap.reset_index(), "cap.json")
    mlflow.log_table(fold_ap_df.reset_index(), "fold_ap.json")
    mlflow.log_table(f1_df.reset_index(names="model"), "best_f1.json")
    mlflow.log_table(paired.reset_index(), "paired.json")
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)